In [21]:
import numpy as np
import pandas as pd
import os
import json
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
ARTIFACTS_DIR =  Path.cwd().parent.joinpath("artifacts")

In [19]:
embeddings = np.load(os.path.join(ARTIFACTS_DIR, "embeddings.npy"))
metadata = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "metadata.parquet"))
config = json.load(open(os.path.join(ARTIFACTS_DIR, "embedding_config.json"), "r"))

print("Embeddings: ", embeddings.shape)
print("Metadata: ", metadata.shape)
print("Configuration: ", config)

Embeddings:  (248218, 384)
Metadata:  (248218, 9)
Configuration:  {'embedding_model': 'BAAI/bge-small-en-v1.5', 'embedding_dimension': 384, 'normalized': True, 'batch_size': 128, 'num_documents': 248218, 'text_columns': ['name', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 'Substitutes', 'Side Effects', 'Uses']}


In [22]:
model = SentenceTransformer(config["embedding_model"], device = "cuda" if torch.cuda.is_available() else "cpu")
print("Model:", model)
print("Device:", model.device)
print("Embedding dimension:", model.get_embedding_dimension())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12367.26it/s]


Model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
Device: cuda:0
Embedding dimension: 384


In [25]:
def retrieve(query, top_k = 5):
    query = model.encode(query)

    scores = cosine_similarity(query.reshape(1, -1), embeddings)[0]

    top_k = np.argsort(scores)[::-1][:top_k]

    results = metadata.iloc[top_k].copy()
    results["similarity"] = scores[top_k]

    return results.reset_index(drop = True)

In [26]:
query = "What is the chemical class of Allegra?"

results = retrieve(query, top_k = 3)

display(results)

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses,document,similarity
0,allegra 30mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Altiva 30mg Tablet, Ultigra 30mg Tablet, Delpo...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,name: allegra 30mg tablet Chemical Class: Diph...,0.765620
1,allegra 180mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex 180 Tablet, Fexofen 180mg Tablet, Mavife...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,name: allegra 180mg tablet Chemical Class: Dip...,0.749719
2,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,name: allegra 120mg tablet Chemical Class: Dip...,0.746621


In [27]:
def search(query, top_k=5):

    results = retrieve(query, top_k = top_k)

    for i, row in results.iterrows():

        print("=" * 80)
        print(f"RESULT #{i + 1}")
        print(f"Similarity: {row['similarity']:.4f}")
        print()

        print(row["document"])

In [28]:
test_queries = [
    "What are alternatives to paracetamol?",
    "What are the side effects of ibuprofen?",
    "Which medicines are used for pain relief?",
    "What drugs are similar to aspirin?",
    "Which medicines can cause nausea?"
]

for query in test_queries:

    print("\n\n")
    print("#" * 100)
    print("QUERY:", query)

    search(query, top_k=3)




####################################################################################################
QUERY: What are alternatives to paracetamol?
RESULT #1
Similarity: 0.6831

name: parafen forte tablet Habit Forming: No Therapeutic Class: PAIN ANALGESICS Substitutes: Answell 400 mg/325 mg Tablet, Bruace 400 mg/325 mg Tablet, Rupar 400 mg/325 mg Tablet, Brufamol Tablet, Zupar 400mg/325mg Tablet Side Effects: Heartburn, Indigestion, Nausea, Stomach pain Uses: Pain relief, Treatment of Fever
RESULT #2
Similarity: 0.6812

name: dr best paracetamol 250 oral suspension Chemical Class: P-Aminophenol Derivative Habit Forming: No Therapeutic Class: PAIN ANALGESICS Action Class: Analgesic & Antipyretic-PCM Substitutes: Tifmol Oral Suspension, Moltrex Oral Suspension, Kyomol Suspension, Nettmol 250mg Oral Suspension, Padicaf 250mg Oral Suspension Side Effects: Indigestion, Stomach pain, Nausea, Vomiting Uses: Pain relief, Treatment of Fever
RESULT #3
Similarity: 0.6807

name: parafen syrup Ha